<a href="https://colab.research.google.com/github/rfroiman/DI_Bootcamp/blob/main/Week10/Day1/Daily_Challenge_LangChain_OpenSource.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Daily Challenge: LangChain Pipelines with Open-Source LLMs (Student)
Use this guided notebook with TODOs. Runs on CPU with small HF models (e.g., flan-t5-small).

## What you'll learn
- Set up LangChain with lightweight open-source models.
- Build an LLMChain using a prompt template.
- Compose a two-step Runnable pipeline (summary ? bullets).
- Bonus: add a simple conversation chain with memory.

## What you will create
- Installed environment for LangChain + transformers.
- LLMChain that rewrites text in a simpler style.
- Runnable pipeline that summarizes then bullet-izes text.
- (Bonus) Conversation chain showing memory.

## Part 1: Environment setup (fast)
Install needed packages. CPU is fine for tiny models.

In [6]:



# Part 1: Inspect the clean Colab environment

import sys
import importlib.metadata as metadata

packages = [
    "requests",
    "huggingface-hub",
    "transformers",
    "langchain",
    "langchain-classic",
    "langchain-community",
    "torch"
]

print("Python:", sys.version.split()[0])
print("-" * 45)

for package in packages:
    try:
        version = metadata.version(package)
        print(f"{package}: {version}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")


Python: 3.13.18
---------------------------------------------
requests: 2.32.4
huggingface-hub: 1.33.0
transformers: 5.22.0
langchain: 1.4.7
langchain-classic: NOT INSTALLED
langchain-community: NOT INSTALLED
torch: 2.11.0+cpu


In [24]:







# Part 1: Restore required dependencies after Colab restart

%pip install -q --no-deps "langchain-classic==1.0.8" "langchain-community==0.4.2"
%pip install -q "sentencepiece" "accelerate"
%pip install -q "langchain-text-splitters>=1.0,<2.0"

print("Dependency installation completed.")



Dependency installation completed.


In [26]:

# Part 1: Verify the essential libraries

import importlib.metadata as metadata

for package in [
    "requests",
    "transformers",
    "huggingface-hub",
    "langchain",
    "langchain-classic",
    "langchain-community"
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")


requests: 2.32.4
transformers: 5.22.0
huggingface-hub: 1.33.0
langchain: 1.4.7
langchain-classic: 1.0.8
langchain-community: 0.4.2


## Part 2: Load a tiny model and build your first LLMChain
Use a small model (e.g., google/flan-t5-small) to keep inference quick.

In [11]:


# Part 2: Verify LangChain and Transformers imports

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains import LLMChain
from langchain_community.llms import HuggingFacePipeline

print("Transformers imports: OK")
print("PromptTemplate: OK")
print("StrOutputParser: OK")
print("LLMChain: OK")
print("HuggingFacePipeline: OK")



Transformers imports: OK
PromptTemplate: OK
StrOutputParser: OK
LLMChain: OK
HuggingFacePipeline: OK


/tmp/ipykernel_585/2253642110.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFacePipeline


In [12]:

# Part 2: Choose a small CPU-friendly model

model_name = "google/flan-t5-small"

print("Selected model:", model_name)


Selected model: google/flan-t5-small


In [13]:


# Part 2: Load the tokenizer and model on CPU

import time
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

start_time = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to("cpu")

load_time = time.perf_counter() - start_time

print("Model:", model_name)
print("Device:", model.device)
print(f"Loading time: {load_time:.2f} seconds")
print("Model loaded successfully.")



config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model: google/flan-t5-small
Device: cpu
Loading time: 12.03 seconds
Model loaded successfully.


In [14]:



# Part 2: Create a compatible generation pipeline for FLAN-T5

from transformers.pipelines.base import Pipeline
from langchain_community.llms import HuggingFacePipeline

class FlanT5Pipeline(Pipeline):
    def _sanitize_parameters(self, **kwargs):
        return {}, kwargs, {}

    def preprocess(self, inputs, **kwargs):
        return self.tokenizer(
            inputs,
            return_tensors="pt",
            truncation=True,
            max_length=512
        )

    def _forward(self, model_inputs, **kwargs):
        generated_ids = self.model.generate(
            **model_inputs,
            max_new_tokens=80,
            do_sample=False
        )
        return {"generated_ids": generated_ids}

    def postprocess(self, model_outputs, **kwargs):
        generated_text = self.tokenizer.decode(
            model_outputs["generated_ids"][0],
            skip_special_tokens=True
        )
        return [{"generated_text": generated_text}]


hf_pipeline = FlanT5Pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text2text-generation",
    framework="pt",
    device=-1
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

print("Custom FLAN-T5 pipeline created successfully.")
print("LangChain wrapper created successfully.")
print("Model:", model_name)
print("Task:", hf_pipeline.task)


Custom FLAN-T5 pipeline created successfully.
LangChain wrapper created successfully.
Model: google/flan-t5-small
Task: text2text-generation


/tmp/ipykernel_585/998208791.py:42: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=hf_pipeline)


In [15]:


# TODO: build prompt + LLMChain for friendly rewriting

import time

template = (
    "Rewrite the following sentence using very simple English "
    "that a beginner can understand. Use common words and "
    "avoid technical terms.\n\n"
    "Sentence: {text}\n\n"
    "Simple version:"
)

prompt = PromptTemplate(
    template=template,
    input_variables=["text"]
)

chain = LLMChain(prompt=prompt, llm=llm)

sample_text = (
    "The individual was unable to comprehend "
    "the instructions provided by the instructor."
)

start_time = time.perf_counter()

rewritten = chain.invoke({"text": sample_text})

inference_time = time.perf_counter() - start_time

print("Original text:", sample_text)
print("Rewritten text:", rewritten["text"])
print(f"Inference time: {inference_time:.2f} seconds")

# Diagnostic: Test FLAN-T5 directly without LangChain

test_input = "Translate English to German: The house is beautiful."

inputs = tokenizer(
    test_input,
    return_tensors="pt"
)

outputs = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False
)

direct_response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

# Diagnostic: Test the model directly without LangChain

test_input = "Translate English to German: The house is beautiful."

inputs = tokenizer(test_input, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50)

direct_response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Direct model test:", test_input)
print("Direct model response:", direct_response)


# Test simplification directly without LangChain

simple_prompt = (
    "Simplify: The individual was unable to comprehend "
    "the instructions provided by the instructor."
)

inputs = tokenizer(simple_prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=80)

simple_response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Direct simplification response:", simple_response)


# Diagnostic: Test text simplification directly with FLAN-T5

simple_prompt = (
    "Simplify: The individual was unable to comprehend "
    "the instructions provided by the instructor."
)

inputs = tokenizer(simple_prompt, return_tensors="pt")

outputs = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=False
)

simple_response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

/tmp/ipykernel_585/2319301524.py:18: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(prompt=prompt, llm=llm)


Original text: The individual was unable to comprehend the instructions provided by the instructor.
Rewritten text: The individual was unable to comprehend the instructions provided by the instructor.
Inference time: 0.69 seconds
Direct model test: Translate English to German: The house is beautiful.
Direct model response: Das Haus ist schön.
Direct simplification response: The instructor was unable to comprehend the instructions provided by the instructor.


## Part 3: Two-step pipeline (summary ? bullets)
Summarize a paragraph, then turn it into 3 bullets using the same LLM.

In [20]:

# Part 3: Define prompts for a two-step pipeline

from langchain_core.prompts import PromptTemplate

summary_prompt = PromptTemplate(
    template=(
        "Summarize the following paragraph in one short, "
        "clear sentence.\n\n"
        "Paragraph: {paragraph}\n\n"
        "Summary:"
    ),
    input_variables=["paragraph"],
)

bullets_prompt = PromptTemplate(
    template=(
        "Extract three key points from this text. "
        "Write each point on a separate line starting with '-'.\n\n"
        "Text: {summary}\n\n"
        "Key points:\n"
        "-"
    ),
    input_variables=["summary"],
)

print("Summary prompt created successfully.")
print("Bullet points prompt created successfully.")


Summary prompt created successfully.
Bullet points prompt created successfully.


In [21]:

# Part 3: Build a two-step Runnable pipeline

# First stage: paragraph -> summary
summary_chain = summary_prompt | llm

# Full pipeline:
# 1. Receive a paragraph
# 2. Generate a summary
# 3. Pass the summary to the second prompt
# 4. Generate three bullet points

summarize_then_bullets = (
    {"summary": summary_chain}
    | bullets_prompt
    | llm
)

print("Summary chain created successfully.")
print("Two-step pipeline created successfully.")


Summary chain created successfully.
Two-step pipeline created successfully.


In [22]:

# Part 3: Test the two-step pipeline

import time

paragraph = """
LangChain is a framework for building applications with
large language models by composing prompts, models, and tools.
It supports chains, agents, and retrieval workflows.
"""

start_time = time.perf_counter()

bullets_output = summarize_then_bullets.invoke({
    "paragraph": paragraph
})

pipeline_time = time.perf_counter() - start_time

print("Original paragraph:", paragraph.strip())
print("\nGenerated bullet points:")
print(bullets_output)
print(f"\nPipeline execution time: {pipeline_time:.2f} seconds")


# Part 3: Inspect each pipeline stage separately

summary_result = summary_chain.invoke({
    "paragraph": paragraph
})

bullets_result = (bullets_prompt | llm).invoke({
    "summary": summary_result
})

print("\n--- Pipeline Diagnostics ---")
print("Stage 1 - Summary:")
print(summary_result)

print("\nStage 2 - Three bullet points:")
print(bullets_result)

bullet_lines = [
    line.strip()
    for line in bullets_result.splitlines()
    if line.strip().startswith(("-", "*", "•"))
]

print("\nBullet points detected:", len(bullet_lines))
print("Expected bullet points: 3")



Original paragraph: LangChain is a framework for building applications with
large language models by composing prompts, models, and tools.
It supports chains, agents, and retrieval workflows.

Generated bullet points:
LangChain is a framework for building applications with large language models by composing prompts, models, and tools.

Pipeline execution time: 1.46 seconds

--- Pipeline Diagnostics ---
Stage 1 - Summary:
LangChain is a framework for building applications with large language models by composing prompts, models, and tools.

Stage 2 - Three bullet points:
LangChain is a framework for building applications with large language models by composing prompts, models, and tools.

Bullet points detected: 0
Expected bullet points: 3


## Part 4 (Bonus): Conversation chain with memory
Show how two turns keep context.

In [25]:


# Part 4: Build a simple conversation chain with memory

from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory

# Create conversation memory
memory = ConversationBufferMemory(
    memory_key="history",
    return_messages=False
)

# Create the conversation chain
convo = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False
)

# First conversation turn
reply1 = convo.predict(
    input="Hi there! What's LangChain?"
)

# Second conversation turn
reply2 = convo.predict(
    input="Can it help me build a simple chatbot?"
)

print("Turn 1:", reply1)
print("Turn 2:", reply2)

# Inspect stored conversation history
print("\n--- Conversation Memory ---")
print(memory.load_memory_variables({})["history"])



/tmp/ipykernel_585/2427728790.py:7: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(
/usr/local/lib/python3.13/dist-packages/pydantic/main.py:280: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
/tmp/ipykernel_585/2427728790.py:13: LangChainDeprecationWarning: The class `Convers

Turn 1: LangChain is a fictional character in the "LangChain" series.
Turn 2: It's a simple chatbot.

--- Conversation Memory ---
Human: Hi there! What's LangChain?
AI: LangChain is a fictional character in the "LangChain" series.
Human: Can it help me build a simple chatbot?
AI: It's a simple chatbot.


## Your observations

- **Latency:** The `google/flan-t5-small` model initially loaded in 12.03 seconds on CPU. Text rewriting inference took approximately 0.63–1.39 seconds across the recorded tests. The two-step summarization and bullet-point pipeline completed in 1.46 seconds in the final test.

- **Quality:** The model successfully performed English-to-German translation and generated a short summary. However, it repeatedly returned the original text instead of simplifying it. The two-step pipeline successfully executed both stages but failed to produce the requested three bullet points (0/3 detected). The conversation memory correctly stored two interactions, although the generated responses were inaccurate or overly simplistic.

- **Quirks:** The model hallucinated that LangChain was a fictional character. It also produced a misleading simplification that changed the original meaning. Transformers 5.22.0 removed the `text2text-generation` pipeline, requiring a custom adapter for FLAN-T5. Several LangChain components generated deprecation warnings. Overall, the model was lightweight and reasonably fast on CPU, but its instruction-following ability and response accuracy were limited.